# シミュレーションとコンピュータグラフィックス

シミュレーションは「状態遷移の正しさ」、CGは「観測の見え方」を担います。世界モデルではこの2層を分けて考えることが実装上の要点です。


## 背景と目的

物理更新が正しくても描画が不適切だと学習器には誤った観測が入ります。

逆に描画が高品質でも遷移モデルが誤っていれば計画は壊れます。

簡単な投射運動をシミュレーションし、2Dグリッドへレンダリングします。


## 最初に解きたい疑問

1. `シミュレーション` と `CG` は、どちらが「世界の動き」でどちらが「見え方」なのか。
2. どうして物理更新と描画を分ける必要があるのか。
3. `vy *= -0.4` は何を表しているのか。
4. `canvas` に `*` を置く処理は、何を近似しているのか。
5. `sim-to-real` 調整では、何を直すのか。


## 先に押さえる言葉

- `シミュレーション`: 状態が時間とともにどう変わるかを計算で再現することです。物理や環境の動きを扱います。
- `CG`: 連続的な状態を画面上の見た目に変換することです。学習器に入る観測の形を作ります。
- `投射運動`: 重力のある中で放物線を描く運動です。ここではボールのような動きの例として使っています。
- `sim-to-real`: シミュレーションで学んだものを現実に持っていくときのズレ対策です。物理と見え方の差を詰めます。


## 実行前の見取り図

1. 最初の 5 状態が `first 5 simulated states` に出ているか。
2. `canvas` の `*` が放物線っぽく並んでいるか。
3. 床に当たったあとに `vy` が反転して跳ね返っているか。


## 出力の読み方

- `canvas` の星形が放物線に見えるかをどう確認するか。
- 最初の数ステップだけ見て何を判断するか。


## つまずきやすい点

- 座標をグリッドに丸めると何が失われるかの説明。
- 反発係数 `-0.4` の意味を、初学者に分かる言い方で補足する説明。
- 丸めで失う情報の説明。
- `-0.4` の物理的意味を初学者向けに言い換える説明。


## このノートの守備範囲

このノートでは次の点は入口だけ触れるか、別ノートに分けて扱います。

- ここでの物理は最小の toy model で、実物理エンジンではない点。


## コード 1: 反復の進み方を観察する

このセルでは 反復の進み方を観察する ための最小コードを動かします。 実行時は「最初の 5 状態が `first 5 simulated states` に出ているか。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
import numpy as np

dt = 0.1
g = -9.8
x, y = 0.0, 0.0
vx, vy = 2.0, 6.0
traj = []
for _ in range(25):
    x += vx * dt
    y += vy * dt
    vy += g * dt
    if y < 0:
        y = 0
        vy *= -0.4
    traj.append((x, y))
print('first 5 simulated states =', [(round(a,2), round(b,2)) for a,b in traj[:5]])


## コード 2: 反復の進み方を観察する

このセルでは 反復の進み方を観察する ための最小コードを動かします。 実行時は「`canvas` の `*` が放物線っぽく並んでいるか。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
# CG: 連続状態をグリッドに投影して可視化
W, H = 40, 12
canvas = [['.' for _ in range(W)] for _ in range(H)]

for x, y in traj:
    px = min(W - 1, max(0, int(x * 3)))
    py = min(H - 1, max(0, int(y * 1.5)))
    canvas[H - 1 - py][px] = '*'

for row in canvas:
    print(''.join(row))


遷移モデルと描画モデルを分離しておくと、sim-to-real調整で「物理を直すべきか、見え方を直すべきか」を切り分けやすくなります。
